<a href="https://colab.research.google.com/github/Naganarthanan/RiceGuard-DL/blob/naga/Model1_Custom_CNN_2D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

drive_target = '/content/drive/MyDrive/RiceGuard_DL/rice_disease_split'
print("Contents:", os.listdir(drive_target))

for split in ['train', 'val', 'test']:
    path = os.path.join(drive_target, split)
    print(f"{split}: {sorted(os.listdir(path))}")

Contents: ['test', 'val', 'train']
train: ['Bacterial_Leaf_Blight', 'Brown_Spot', 'Healthy_Rice_Leaf', 'Leaf_Blast', 'Leaf_scald', 'Sheath_Blight']
val: ['Bacterial_Leaf_Blight', 'Brown_Spot', 'Healthy_Rice_Leaf', 'Leaf_Blast', 'Leaf_scald', 'Sheath_Blight']
test: ['Bacterial_Leaf_Blight', 'Brown_Spot', 'Healthy_Rice_Leaf', 'Leaf_Blast', 'Leaf_scald', 'Sheath_Blight']


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = f'{drive_target}/train'
val_dir = f'{drive_target}/val'
test_dir = f'{drive_target}/test'

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_test_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = val_test_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

num_classes = len(train_data.class_indices)
print("Classes:", train_data.class_indices)
print("Number of classes:", num_classes)
print("Train samples:", train_data.samples)
print("Val samples:", val_data.samples)
print("Test samples:", test_data.samples)

Found 2655 images belonging to 6 classes.
Found 567 images belonging to 6 classes.
Found 576 images belonging to 6 classes.
Classes: {'Bacterial_Leaf_Blight': 0, 'Brown_Spot': 1, 'Healthy_Rice_Leaf': 2, 'Leaf_Blast': 3, 'Leaf_scald': 4, 'Sheath_Blight': 5}
Number of classes: 6
Train samples: 2655
Val samples: 567
Test samples: 576


In [ ]:
from tensorflow.keras import layers, models

custom_cnn = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

custom_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

custom_cnn.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,734 (42.61 MB)

 Trainable params: 11,169,734 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import shutil
import time

start = time.time()

# Drive-ல இருந்து local Colab disk-க்கு copy பண்றது
local_path = '/content/rice_disease_split'
shutil.copytree('/content/drive/MyDrive/RiceGuard_DL/rice_disease_split', local_path)

end = time.time()
print(f"Copy time: {(end-start)/60:.2f} minutes")
print("Copied to:", local_path)

Copy time: 13.82 minutes
Copied to: /content/rice_disease_split


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = f'{local_path}/train'
val_dir = f'{local_path}/val'
test_dir = f'{local_path}/test'

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_test_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = val_test_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print("Train:", train_data.samples)
print("Val:", val_data.samples)
print("Test:", test_data.samples)

Found 2655 images belonging to 6 classes.
Found 567 images belonging to 6 classes.
Found 576 images belonging to 6 classes.
Train: 2655
Val: 567
Test: 576


In [9]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('custom_cnn_best.h5', monitor='val_accuracy', save_best_only=True)

history = custom_cnn.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 561ms/step - accuracy: 0.6925 - loss: 0.8109

83/83 ━━━━━━━━━━━━━━━━━━━━ 52s 622ms/step - accuracy: 0.7032 - loss: 0.7913 - val_accuracy: 0.6914 - val_loss: 0.8014
Epoch 2/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 562ms/step - accuracy: 0.7051 - loss: 0.7920

83/83 ━━━━━━━━━━━━━━━━━━━━ 51s 613ms/step - accuracy: 0.7104 - loss: 0.7635 - val_accuracy: 0.7707 - val_loss: 0.6779
Epoch 3/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 51s 615ms/step - accuracy: 0.7296 - loss: 0.7209 - val_accuracy: 0.7672 - val_loss: 0.6610
Epoch 4/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 50s 597ms/step - accuracy: 0.7424 - loss: 0.6852 - val_accuracy: 0.7496 - val_loss: 0.7003
Epoch 5/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 50s 605ms/step - accuracy: 0.7492 - loss: 0.6816 - val_accuracy: 0.7478 - val_loss: 0.6734
Epoch 6/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 51s 612ms/step - accuracy: 0.7548 - loss: 0.6662 - val_accuracy: 0.7654 - val_loss: 0.6530
Epoch 7/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 50s 601ms/step - accuracy: 0.7529 - loss: 0.6400 - val_accuracy: 0.7513 - val_loss: 0.6133
Epoch 8/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 51s 620ms/step - accuracy: 0.7593 - loss: 0.6280 - val_accuracy: 0.7372 - val_loss: 0.6322
Epoch 9/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 552ms/step - accuracy: 0.7579 - loss: 0.6382

83/83 ━━━━━━━━━━━━━━━━━━━━ 51s 613ms/step - accuracy: 0.7657 - loss: 0.6140 - val_accuracy: 0.8060 - val_loss: 0.5970
Epoch 10/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 50s 601ms/step - accuracy: 0.7733 - loss: 0.5937 - val_accuracy: 0.7601 - val_loss: 0.6216
Epoch 11/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 52s 624ms/step - accuracy: 0.7849 - loss: 0.5786 - val_accuracy: 0.7919 - val_loss: 0.5421
Epoch 12/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 81s 616ms/step - accuracy: 0.7740 - loss: 0.5852 - val_accuracy: 0.7848 - val_loss: 0.5766
Epoch 13/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 50s 604ms/step - accuracy: 0.7887 - loss: 0.5821 - val_accuracy: 0.8007 - val_loss: 0.5728
Epoch 14/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 50s 604ms/step - accuracy: 0.7917 - loss: 0.5403 - val_accuracy: 0.7919 - val_loss: 0.5791
Epoch 15/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 51s 615ms/step - accuracy: 0.7883 - loss: 0.5481 - val_accuracy: 0.8060 - val_loss: 0.5330


In [10]:
model_save_path = f'{drive_target}/custom_cnn_best.h5'
custom_cnn.save(model_save_path)
print(f"Model saved to: {model_save_path}")

Model saved to: /content/drive/MyDrive/RiceGuard_DL/rice_disease_split/custom_cnn_best.h5
